# M04 · Model from a step test, tune it, and cascade the loops

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/04_control_motion/M04_model_from_a_step_test_tune_it_and_cascade_the_l.ipynb)

**Track:** Control and motion · **Skill axis (sheet):** Control and motion · **Time:** about 60 min

**Skill Hub concepts:** First-order plus dead-time model; Cascade control

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("M04")

## Map: step test -> model -> tuning -> loop structure

**First-order plus dead time (FOPDT):** $G(s) = \dfrac{K e^{-\theta s}}{\tau s + 1}$. Step response for an input
step $\Delta u$: $y(t) = K\Delta u\,(1 - e^{-(t-\theta)/\tau})$ for $t \ge \theta$.

Two-point fit: find when $y$ reaches 28.3% and 63.2% of its total change:
$$\tau = 1.5\,(t_{63} - t_{28}), \qquad \theta = t_{63} - \tau, \qquad K = \Delta y/\Delta u$$

SIMC PI tuning (Skogestad), closed-loop time constant $\tau_c$ (default $\tau_c = \theta$):
$$K_c = \frac{\tau}{K(\tau_c + \theta)}, \qquad T_i = \min\big(\tau,\ 4(\tau_c + \theta)\big)$$

**Cascade:**
```
 pos_sp ─(+)─ P ──> vel_sp (clamped to v_max) ─(+)─ PI ──> current (clamped to i_max) ──> motor ──┬─> vel ──∫──> pos
          (-)                                    (-)                                              │
           └────────────────── pos ───────────────┴────────────── vel ───────────────────────────┘
```
The inner loop must be several times faster than the outer one; it rejects load disturbances before they show up
in position, and the clamp on its setpoint becomes a clean velocity limit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Task 1: fit an FOPDT model from step data
`t`, `y` are samples starting at the step (y starts at its initial value), `du` the input step. Use linear
interpolation to find the crossing times.

In [ ]:
def fit_fopdt(t, y, du):
    raise NotImplementedError  # TODO

lab.check("fit", fit_fopdt)

In [ ]:
%%writefile m04_fit.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/m04.h
// Grade it with lab.check("fit", "m04_fit.c"). For C++, name the file .cpp in both places.
#include "m04.h"

void fit_fopdt(const double *t, const double *y, size_t n, double du, double *K, double *tau, double *theta) {
    (void)t; (void)y; (void)n; (void)du;
    *K = 0; *tau = 0; *theta = 0; /* TODO: interpolate the 28.3 % and 63.2 % crossing times */
}

In [ ]:
# lab.check("fit", "m04_fit.c")  # uncomment after writing the C/C++ cell above

## Task 2: SIMC PI settings

In [ ]:
def simc_pi(K, tau, theta, tau_c=None):
    raise NotImplementedError  # TODO

lab.check("simc", simc_pi)

In [ ]:
%%writefile m04_simc.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/m04.h
// Grade it with lab.check("simc", "m04_simc.c"). For C++, name the file .cpp in both places.
#include "m04.h"

void simc_pi(double K, double tau, double theta, double tau_c, double *kc, double *ti) {
    (void)K; (void)tau; (void)theta; (void)tau_c;
    *kc = 0; *ti = 0; /* TODO */
}

In [ ]:
# lab.check("simc", "m04_simc.c")  # uncomment after writing the C/C++ cell above

## Task 3: a position-over-velocity cascade
Motor: $\dot\omega = 10\,i - \omega$ (rad/s, current $i$ in amperes, $|i| \le$ `i_max`). Outer: P on position gives the
velocity setpoint, clamped to `v_max`. Inner: PI on velocity gives the current, clamped, with conditional integration.

In [ ]:
class Cascade:
    def __init__(self, kp_pos, kp_vel, ki_vel, v_max, i_max, dt):
        self.kp_pos, self.kp_vel, self.ki_vel = kp_pos, kp_vel, ki_vel
        self.v_max, self.i_max, self.dt, self.integ = v_max, i_max, dt, 0.0

    def update(self, pos_sp, pos, vel):
        raise NotImplementedError  # TODO

lab.check("cascade", Cascade)

In [ ]:
%%writefile m04_cascade.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/m04.h
// Grade it with lab.check("cascade", "m04_cascade.c"). For C++, name the file .cpp in both places.
#include "m04.h"

void cascade_init(Cascade *c, double kp_pos, double kp_vel, double ki_vel, double v_max, double i_max, double dt) {
    c->kp_pos = kp_pos; c->kp_vel = kp_vel; c->ki_vel = ki_vel;
    c->v_max = v_max; c->i_max = i_max; c->dt = dt; c->integ = 0;
}

double cascade_update(Cascade *c, double pos_sp, double pos, double vel) {
    (void)c; (void)pos_sp; (void)pos; (void)vel;
    return 0; /* TODO: P on position -> clamped velocity setpoint -> PI on velocity with anti-windup -> clamp */
}

In [ ]:
# lab.check("cascade", "m04_cascade.c")  # uncomment after writing the C/C++ cell above

In [ ]:
def run_cascade(ctrl, target=3.0, dt=1e-3, seconds=3.0, load=0.0):
    pos = vel = 0.0
    log = []
    for k in range(int(seconds / dt)):
        i = ctrl.update(target, pos, vel)
        vel += (10 * i - vel - (load if k * dt > 2.0 else 0.0)) * dt
        pos += vel * dt
        log.append((k * dt, pos, vel, i))
    return np.array(log)

try:
    L = run_cascade(Cascade(6.0, 1.5, 8.0, 4.0, 2.0, 1e-3), load=5.0)
    fig, ax = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
    for k, name in enumerate(["position (rad)", "velocity (rad/s)", "current (A)"]):
        ax[k].plot(L[:, 0], L[:, k + 1]); ax[k].set_ylabel(name)
    ax[2].set_xlabel("s  (a load disturbance starts at 2 s)")
    plt.show()
except NotImplementedError:
    print("finish Task 3 first")

## Checkpoint: Math
**M1 (Apply).** Step test: 28.3% reached at 1.2 s, 63.2% at 2.0 s. Time constant $\tau$ in seconds?

**M2 (Apply).** SIMC with $K = 2$, $\tau = 1.2$ s, $\theta = 0.8$ s, $\tau_c = \theta$. Controller gain $K_c$?

**M3 (Apply).** Same case: integral time $T_i$ in seconds?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** Why does the cascade reject the load disturbance at 2 s better than a single position PID?
- a) it has more gains to tune
- b) the fast inner velocity loop sees the disturbance as a velocity error immediately and corrects it before much position error builds up
- c) the outer loop is faster

**K2 (Analyze).** Explain why dead time $\theta$ limits how fast any feedback loop can be made, using the SIMC rule.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `m04.h`).
3. In `roboprep/checks/m04.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()